In [2]:
pip install webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Setup ChromeDriver automatically
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")  # Optional: opens browser maximized

driver = webdriver.Chrome(service=service, options=options)

# Load the MPUDC tenders page
url = "https://www.tenderdetail.com/Indian-Tender/Mpudc"
driver.get(url)
time.sleep(5)  # Allow time to load

# Scroll to load more tenders (lazy loading)
for _ in range(50):  # Increase range for more records
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

# Extract tender information
tenders = []
tender_blocks = driver.find_elements(By.CLASS_NAME, "tender_row")

for block in tender_blocks:
    try:
        title = block.find_element(By.CLASS_NAME, "m-brief").text.strip()
        tdr_id = block.find_element(By.CLASS_NAME, "m-tender-id").text.strip()
        due_date = block.find_element(By.CLASS_NAME, "m-due-date").find_element(By.XPATH, "..").text.strip()
        value = block.find_element(By.CLASS_NAME, "m-value").find_element(By.XPATH, "..").text.strip()
        location = block.find_element(By.CLASS_NAME, "workDesc").text.strip().split("-")[-1].strip()

        tenders.append({
            "TDR ID": tdr_id,
            "Title": title,
            "Location": location,
            "Due Date": due_date,
            "Tender Value": value
        })
    except Exception as e:
        print(f"Error parsing a block: {e}")

# Save to Excel
df = pd.DataFrame(tenders)
df.to_excel("mpudc_tenders.xlsx", index=False)
print("Scraping completed. Data saved to mpudc_tenders.xlsx")

driver.quit()


Error parsing a block: Message: no such element: Unable to locate element: {"method":"css selector","selector":".m-brief"}
  (Session info: chrome=137.0.7151.69); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x0x1133763+63299]
	GetHandleVerifier [0x0x11337a4+63364]
	(No symbol) [0x0xf61113]
	(No symbol) [0x0xfa987e]
	(No symbol) [0x0xfa9c1b]
	(No symbol) [0x0xf9efb1]
	(No symbol) [0x0xfce5c4]
	(No symbol) [0x0xf9eed4]
	(No symbol) [0x0xfce7f4]
	(No symbol) [0x0xfefa4a]
	(No symbol) [0x0xfce376]
	(No symbol) [0x0xf9d6e0]
	(No symbol) [0x0xf9e544]
	GetHandleVerifier [0x0x138e033+2531347]
	GetHandleVerifier [0x0x1389332+2511634]
	GetHandleVerifier [0x0x1159eda+220858]
	GetHandleVerifier [0x0x114a528+156936]
	GetHandleVerifier [0x0x1150c5d+183357]
	GetHandleVerifier [0x0x113b6c8+95912]
	GetHandleVerifier [0x0x113b870+96336]
	GetHandleVerifier [0x0x112664a+9770

In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Setup headless browser using webdriver-manager
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# Automatically download and setup ChromeDriver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# Target URL
url = "https://www.tenderdetail.com/government-tenders/madhya-pradesh-urban-development-company-limited-tenders/1?agid=91046"
driver.get(url)
time.sleep(5)  # Wait for JS to load

# Parse tenders
tenders = driver.find_elements(By.CLASS_NAME, 'tender_row')
data = []

for tender in tenders:
    try:
        tender_id = tender.find_element(By.CLASS_NAME, 'm-tender-id').text.strip()
        description = tender.find_element(By.CLASS_NAME, 'm-brief').text.strip()
        link = tender.find_element(By.CLASS_NAME, 'm-brief').get_attribute('href')
        location_info = tender.find_element(By.TAG_NAME, 'strong').text.strip()
        city, state = [x.strip() for x in location_info.split('-')[-2:]]
        day = tender.find_element(By.CLASS_NAME, 'day').text.strip()
        month = tender.find_element(By.CLASS_NAME, 'month').text.strip()
        year = tender.find_element(By.CLASS_NAME, 'year').text.strip()
        due_date = f"{day} {month} {year}"
        value_elem = tender.find_elements(By.CLASS_NAME, 'tender-value')
        value = value_elem[0].text.strip() if value_elem else "N/A"

        data.append({
            'Tender ID': tender_id,
            'Description': description,
            'City': city,
            'State': state,
            'Due Date': due_date,
            'Tender Value': value,
            'Notice URL': link
        })
    except Exception as e:
        print(f"Error processing a tender: {e}")

driver.quit()

# Export to Excel
df = pd.DataFrame(data)
df.to_excel("MPUDC_Tenders_Live.xlsx", index=False)
print("✅ Scraping complete. Saved to MPUDC_Tenders_Live.xlsx")


✅ Scraping complete. Saved to MPUDC_Tenders_Live.xlsx


In [10]:
import pandas as pd

# Load the Excel file
df = pd.read_excel("MPUDC_Tenders_Live.xlsx")

# Remove the leading tender ID number from the Description
df['Description'] = df['Description'].str.replace(r'^\d+\s+', '', regex=True)

# Save the cleaned version
df.to_excel("MPUDC_Tenders_Cleaned.xlsx", index=False)

print("✅ Cleaned file saved as 'MPUDC_Tenders_Cleaned.xlsx'")


✅ Cleaned file saved as 'MPUDC_Tenders_Cleaned.xlsx'
